# 文件一：环境设置与小文件处理
一. **环境设置**  
  - 1.1 设置数据库上下文  
  - 1.2 检查 Volume
  - 1.3 创建客户维度表（外部表）
  - 1.4 创建产品维度表
  - 1.5 创建订单事实表 

二. **验证生成数据质量**  
  - 2.1 热门客户（数据倾斜）  
  - 2.2 热门产品（数据倾斜）
  - 2.3 日期订单检测（分区爆炸）

三. **DWS & ADS 简易搭建**
 - 3.1 DWS：客户+月份轻度汇总
 - 3.2 ADS：简单客户分群（基于订单数）

四. **小文件问题**  
  - 4.1 检查小文件数量
  - 4.2 小文件处理演示

五. **分区前后测试**
- 5.1 分区表创建
- 5.2 检查分区后文件数
- 5.3 分区前后查询效率对比


In [0]:
--一.环境设置
--1.1 设置数据库上下文
-- 使用 Catalog 和 Schema
USE CATALOG data_catalog;
USE SCHEMA retail_data;

In [0]:
%python
#--1.2 检查 Volume 中是否有文件（数据是以非表形式上传的）
%ls /Volumes/data_catalog/retail_data/retail_volume/
#注：如果数据一开始就是以表形式上传的，可在SQL代码块执行SHOW TABLES查看数据;


customer.csv*    orders_025.csv*  orders_051.csv*  orders_077.csv*
orc_data/        orders_026.csv*  orders_052.csv*  orders_078.csv*
orders_001.csv*  orders_027.csv*  orders_053.csv*  orders_079.csv*
orders_002.csv*  orders_028.csv*  orders_054.csv*  orders_080.csv*
orders_003.csv*  orders_029.csv*  orders_055.csv*  orders_081.csv*
orders_004.csv*  orders_030.csv*  orders_056.csv*  orders_082.csv*
orders_005.csv*  orders_031.csv*  orders_057.csv*  orders_083.csv*
orders_006.csv*  orders_032.csv*  orders_058.csv*  orders_084.csv*
orders_007.csv*  orders_033.csv*  orders_059.csv*  orders_085.csv*
orders_008.csv*  orders_034.csv*  orders_060.csv*  orders_086.csv*
orders_009.csv*  orders_035.csv*  orders_061.csv*  orders_087.csv*
orders_010.csv*  orders_036.csv*  orders_062.csv*  orders_088.csv*
orders_011.csv*  orders_037.csv*  orders_063.csv*  orders_089.csv*
orders_012.csv*  orders_038.csv*  orders_064.csv*  orders_090.csv*
orders_013.csv*  orders_039.csv*  orders_065.csv*  orders_091.

In [0]:
%python
#1.3 创建客户维度表（外部表）
#客户表dim_customer，仅包含customer_id
#注：Unity Catalog 只支持将数据写入 Managed Table（Delta 格式），不支持直接以 CSV 格式注册外部表。因此我们需要使用转换。
# 读取 Volume 中的 customer.csv
df_customer = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/data_catalog/retail_data/retail_volume/customer.csv")

# 写入为 Managed Table（Delta 格式）
df_customer.write.mode("overwrite").saveAsTable("data_catalog.retail_data.dim_customer")

# 验证
display(spark.sql("SELECT * FROM data_catalog.retail_data.dim_customer LIMIT 5"))

customer_id
13021BA
13022BA
13023BA
13024BA
13025BA


In [0]:
%python
#1.4 创建产品维度表
# 产品表：包含 product_id, product_price 等
df_product = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/data_catalog/retail_data/retail_volume/product.csv")

df_product.write.mode("overwrite").saveAsTable("data_catalog.retail_data.dim_product")

# 验证
display(spark.sql("SELECT * FROM data_catalog.retail_data.dim_product LIMIT 5"))

product_id,product_category,product_model,product_name,product_price
214,配件,Easton-Z5,头盔,704
217,配件,Easton-Z5,头盔,164
222,配件,Easton-Z5,头盔,75
225,服装,Baseball Cap,帽子,809
228,服装,Rawlings Long-Sleeve,棒球服,331


In [0]:
%python
#1.5 创建订单事实表
# 订单事实表：使用通配符读取所有orders_*.csv
df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/data_catalog/retail_data/retail_volume/orders_*.csv")

# 写入为 Managed Table（注意：这里会合并所有小文件为一个 Delta 表）
df_orders.write.mode("overwrite").saveAsTable("data_catalog.retail_data.fact_order_raw")

# 验证行数（应该为1000W行）
print(f"Total rows: {df_orders.count()}")

Total rows: 10000000


In [0]:
--二.验证生成数据质量（是否可能存在数据倾斜、分区爆炸）
--2.1 检查热门客户占比（预期 ~30%），cnt=数量，persentage=占比
SELECT 
  CASE WHEN customer_id IN ('13021BA','13022BA','13023BA','13024BA','13025BA',
                            '13026BA','13027BA','13028BA','13029BA','13030BA')
       THEN 'hot_customer' ELSE 'normal' END AS customer_type,
  COUNT(*) AS cnt,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_order_raw), 2) AS percentage
FROM fact_order_raw
GROUP BY 1;

customer_type,cnt,percentage
normal,7001683,70.02
hot_customer,2998317,29.98


In [0]:
--2.2 检查热门产品占比（预期 ~50%），cnt为数量，percentage为占比
SELECT 
  CASE WHEN product_id IN (214,310,320,330,529) THEN 'hot_product' ELSE 'normal' END AS product_type,
  COUNT(*) AS cnt,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_order_raw), 2) AS percentage
FROM fact_order_raw
GROUP BY 1;

product_type,cnt,percentage
hot_product,5000119,50.00
normal,4999881,50.00


In [0]:
--2.3 检查分区爆炸日期（预期 2020-01-01, 2020-10-01, 2020-11-11 等远高于普通日期）
SELECT order_date, COUNT(*) AS cnt
FROM fact_order_raw
WHERE order_date IN ('2020-01-01','2020-10-01','2020-11-11','2020-12-25','2021-01-01')
GROUP BY order_date
ORDER BY cnt DESC;

order_date,cnt
2020-12-25,117552
2021-01-01,117061
2020-10-01,116802
2020-01-01,116551
2020-11-11,116280


In [0]:
-- 三.DWS & ADS 层构建
-- 3.1 DWS 层：客户月度汇总表
CREATE OR REPLACE TABLE dws_customer_month_summary
USING DELTA
AS
SELECT 
    customer_id,
    DATE_FORMAT(order_date, 'yyyy-MM') AS month_key,
    COUNT(*) AS order_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount
FROM fact_order_raw
GROUP BY customer_id, month_key;


num_affected_rows,num_inserted_rows


In [0]:
-- 3.2 ADS 层：客户活跃度分层
CREATE OR REPLACE TABLE ads_customer_activity
USING DELTA
AS
SELECT 
    customer_id,
    COUNT(*) AS order_count,
    CASE 
        WHEN COUNT(*) >= 100 THEN 'very_active'
        WHEN COUNT(*) >= 10 THEN 'active'
        ELSE 'low_activity'
    END AS activity_level
FROM fact_order_raw
GROUP BY customer_id;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) AS row_count2
FROM ads_customer_activity;

row_count2
18484


In [0]:
%python
# 四.小文件问题
#4.1 检查小文件数量（不分区）
# 获取 Delta 表真实路径
# 查看表的详细统计信息（文件数、大小等）
# 为什么会有小文件：写入Delta时， Spark可能根据内置并行度对数据进行合并，因此需要检查Delta底层。
# 小文件数量不一定等于上传文件数量，需要分析写入过程。
detail = spark.sql("DESCRIBE DETAIL data_catalog.retail_data.fact_order_raw").collect()[0]
print(f"文件数量: {detail.numFiles}")
print(f"表大小（字节）: {detail.sizeInBytes}")
print(f"位置: {detail.location}")

文件数量: 6
表大小（字节）: 74751510
位置: 


In [0]:
%python
# 4.2 小文件处理演示
#  (1).创建一个临时表demo_optimize
spark.sql("""
    CREATE OR REPLACE TABLE demo_optimize
    USING DELTA
    PARTITIONED BY (dt)
    AS 
    SELECT '2020-01-01' AS dt, * FROM fact_order_raw LIMIT 1000
""")

# (2).再追加两次数据到同一个分区（每次产生一个新文件）
spark.sql("INSERT INTO demo_optimize SELECT '2020-01-01' AS dt, * FROM fact_order_raw LIMIT 1000")
spark.sql("INSERT INTO demo_optimize SELECT '2020-01-01' AS dt, * FROM fact_order_raw LIMIT 1000")

# (3).查看文件数（>1是合理的）
detail = spark.sql("DESCRIBE DETAIL demo_optimize").collect()[0]
print(f"优化前文件数: {detail.numFiles}")


优化前文件数: 3


In [0]:
%python
# (4).执行 OPTIMIZE
spark.sql("OPTIMIZE demo_optimize")

In [0]:
%python
# (5).查看优化后文件数（应该为1）
detail_opt = spark.sql("DESCRIBE DETAIL demo_optimize").collect()[0]
print(f"优化后文件数: {detail_opt.numFiles}")

优化后文件数: 1


In [0]:
%python
# 五.数据分区
# 5.1 分区表创建
# 创建按月分区的表（从原表读取数据，按月份字符串分区）
# 为什么不按日分区：因为一年有365/366天，很容易导致分区数量膨胀，还会导致颗粒度过细(同一分区内只有一个小文件)，无法执行合并优化。
spark.sql("""
    CREATE OR REPLACE TABLE fact_order_month_partitioned
    USING DELTA
    PARTITIONED BY (month_key)
    AS 
    SELECT *, SUBSTR(order_date, 1, 7) AS month_key
    FROM fact_order_raw
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%python
# 5.2 检查分区后文件数
#原始数据有132个月，如果分区后文件数不对（文件数量过多），说明分区产生了额外的小文件问题。
#为什么Spark合并后文件只有6个，仍要分区？因为分区能让特定条件的查询更快，通过对比可以检测出来。
month_before = spark.sql("DESCRIBE DETAIL fact_order_month_partitioned").collect()[0]
print(f"分区后文件数: {month_before.numFiles}")

分区后文件数: 132


In [0]:
%python
#5.3 查询性能对比：检验分区的影响
#对比查询：从 CSV 外部表 vs 分区表查询相同聚合，分别测试运行时长以观察效率差异。
#查询目标：从 raw 表（CSV）统计一定日期范围内每个客户的总金额

import time

# 查询1：raw 表
start = time.time()
spark.sql("""
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_raw
    WHERE order_date >= '2020-01-01' AND order_date <= '2020-12-31'
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
""").show()
end = time.time()
raw_time = end - start

# 查询2：按月分区表
start = time.time()
spark.sql("""
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_month_partitioned
    WHERE month_key BETWEEN '2020-01' AND '2020-12'
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
""").show()
end = time.time()
partition_month_time = end - start

# 打印对比
print(f"Raw 表耗时（无分区）: {raw_time:.2f} 秒")
print(f"分区表（使用month_key条件）: {partition_month_time:.2f} 秒")
print(f"性能提升: {(partition_month_time/raw_time - 1)*100:.1f}%")

#注：对于全表聚合查询（无日期过滤），按月分区表因文件数过多（132个 vs 6个）导致性能略差。
#如果去掉where条件进行查询，查询2（分区表）会比查询1（初始表）更慢。
#这说明分区策略需要与查询模式匹配：分区对范围查询友好，对全表扫描不友好。

+-----------+--------+
|customer_id|   total|
+-----------+--------+
|    13025BA|21838966|
|    13026BA|21827791|
|    13028BA|21702702|
|    13023BA|21604209|
|    13029BA|21596067|
|    13021BA|21569177|
|    13030BA|21542906|
|    13024BA|21483391|
|    13022BA|21398939|
|    13027BA|21375556|
+-----------+--------+

+-----------+--------+
|customer_id|   total|
+-----------+--------+
|    13025BA|21838966|
|    13026BA|21827791|
|    13028BA|21702702|
|    13023BA|21604209|
|    13029BA|21596067|
|    13021BA|21569177|
|    13030BA|21542906|
|    13024BA|21483391|
|    13022BA|21398939|
|    13027BA|21375556|
+-----------+--------+

Raw 表耗时（无分区）: 1.05 秒
分区表（使用month_key条件）: 1.67 秒
性能提升: 59.8%
